In [0]:
webhook_url = dbutils.secrets.get(
    scope="prj-secret",
    key="slack-webhook"
)

print("Secret Loaded Successfully")

In [0]:
import requests
import json

def send_slack_message(message):

    headers = {
        "Content-Type": "application/json"
    }

    payload = {
        "text": message
    }

    response = requests.post(
        webhook_url,
        headers=headers,
        data=json.dumps(payload)
    )

    print(response.status_code)

In [0]:
# Step 1 - Import Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Step 2 - Use Catalog & Schema
spark.sql("USE CATALOG Bronze_Catalog")
spark.sql("USE SCHEMA Bronze_SCH")

In [0]:
# Step 3 - Read Parquet
tariff_df = spark.read.parquet(
    "abfss://rev1@adlsstgacntp2026.dfs.core.windows.net/Parquet_data/tariff_metrics_stream_v2.parquet"
)

In [0]:
# Step 4 - Check Schema
tariff_df.printSchema()

In [0]:
# Step 6 - Add Ingestion Timestamp
from pyspark.sql.functions import current_timestamp

tariff_df = tariff_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

In [0]:
# Step 7 - Verify
tariff_df.printSchema()

In [0]:
# Step 8 - Count Records
print("Total Records :", tariff_df.count())

In [0]:
spark.sql("""
INSERT INTO Bronze_Catalog.Bronze_SCH.Watermark_Table
VALUES (
'Bronze_Tariff_Metrics',
TIMESTAMP('1900-01-01 00:00:00')
);
""")

In [0]:
# Step 9 - Read Watermark
last_watermark = spark.sql("""
SELECT last_processed_timestamp
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
WHERE table_name='Bronze_Tariff_Metrics'
""").collect()[0][0]

print(last_watermark)

In [0]:

# Step 10 - Incremental Filter
incremental_df = tariff_df.filter(
    col("ingestion_timestamp") > last_watermark
)

records_loaded = incremental_df.count()

print(f"New Records : {records_loaded}")

display(incremental_df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp

try:

    incremental_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema","true") \
        .saveAsTable("Bronze_Catalog.Bronze_SCH.Bronze_Tariff_Metrics")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Tariff_Metrics',
            'Tariff Bronze Pipeline',
            current_timestamp(),
            {records_loaded},
            'SUCCESS',
            NULL
        )
    """)

    print(f"✅ Bronze_Tariff_Metrics loaded successfully.")
    print(f"📊 Records Loaded : {records_loaded}")
    send_slack_message(f"""
✅ Bronze_Tariff_Metrics SUCCESS

Pipeline : Tariff Metrics Bronze Pipeline

Records Loaded : {records_loaded}

Status : SUCCESS
""")

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Tariff_Metrics',
            'Tariff Bronze Pipeline',
            current_timestamp(),
            0,
            'FAILED',
            '{error_message}'
        )
    """)

    print("❌ Bronze_Tariff_Metrics Pipeline Failed")
    print(f"⚠️ Error : {error_message}")
    send_slack_message(f"""
❌ Bronze_Tariff_Metrics FAILED

Pipeline : Tariff Metrics Bronze Pipeline

Error : {error_message}

Status : FAILED
""")
    raise

In [0]:
spark.sql("""
UPDATE Bronze_Catalog.Bronze_SCH.Watermark_Table
SET last_processed_timestamp = (
    SELECT MAX(ingestion_timestamp)
    FROM Bronze_Catalog.Bronze_SCH.Bronze_Tariff_Metrics
)
WHERE table_name='Bronze_Tariff_Metrics'
""")

print("✅ Watermark Updated Successfully")

In [0]:
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
WHERE table_name='Bronze_Tariff_Metrics'
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Audit_Log
WHERE table_name='Bronze_Tariff_Metrics'
ORDER BY load_time DESC
LIMIT 5
""").show(truncate=False)